# Predicting U.S. Airline Departure Delays at Scale  
## A Distributed Machine Learning Approach Using PySpark (Apache Spark MLlib)
**Course:** BIA 678 – Big Data Analytics  
**Student:** Achintya  
**Student ID:** 200213266  


# 1. Introduction & Scope

Airline delays create cascading operational and customer impacts across the U.S. National Airspace System (NAS): missed connections, crew and gate conflicts, aircraft repositioning issues, and downstream network disruption. 

While small-scale models can demonstrate predictive feasibility, real operational value typically requires 
(i) learning from *millions* of flights across a full annual cycle and 
(ii) producing *risk-ranked outputs* that help prioritize interventions (e.g., staffing, gate planning, buffer allocation, customer notifications).

This project builds an end-to-end distributed machine learning pipeline to predict **departure delay magnitude and factors that most contribute to it** (DEP_DELAY) using **PySpark** on **Apache Spark**, with emphasis on scalability, leakage-safe ETL, and decision-oriented evaluation. Rather than treating this as a pure accuracy exercise, the analysis focuses on identifying high-risk flights (tail events) that matter most operationally.

---

## 1.1 Project Objective

**Primary objective:**  
Predict the **extent of departure delays (DEP_DELAY)** for U.S. domestic flights.

**Why “extent” (regression) matters:**  
Operational planning often benefits from an estimated delay magnitude (e.g., 5 vs. 55 minutes) rather than only a binary on-time indicator. However, the project also evaluates the model’s ability to flag high-risk flights, where recall within the riskiest segment (Top-*k* risk) becomes especially relevant.

---

## 1.2 Core Challenges

This problem is difficult for three practical reasons:

1. **Scale (Big Data constraint)**  
   With ~6.5–7.1 million flights in a single year, data preparation, feature engineering, and model training require a distributed processing framework for feasible runtime and memory usage.

2. **Skewness and long-tail behavior**  
   Most flights are near on-time, while severe delays occur less frequently but carry disproportionate operational cost. This creates a “long-tail” target distribution and makes naïve metrics (like average error alone) insufficient.

3. **Seasonality and operational variability**  
   Delay patterns change across months, weekdays, and peak travel periods, and are influenced by network congestion, demand surges, and irregular operations. A robust model must generalize across these regime shifts.

---

## 1.3 Project Scope

### In-scope
- **Dataset:** 2019 U.S. domestic flight operations (BTS-aggregated source)  
- **Task:** Predict **DEP_DELAY** with distributed ML
- **Pipeline:** Ingestion → ETL → leakage prevention → EDA → feature engineering → modeling → evaluation → scaling studies
- **Modeling:** Baselines + Gradient-Boosted Trees (GBT) using Spark MLlib
- **Evaluation:** Standard regression metrics plus risk-focused diagnostics (e.g., Top-*k* recall conceptually) and ablation-style comparisons
- **Systems focus:** Scaling up (data %) and scaling out (cluster workers)

### Out-of-scope (explicitly)
- Incorporating external weather feeds (unless added later as future work)
- Real-time streaming inference deployment (batch-only in this report)
- Causal claims (this is predictive modeling, not causal inference)

## 1.4 Contributions (What this project delivers)

This work contributes a reproducible big-data ML workflow with:

- A leakage-safe ETL design (ensuring only pre-departure / allowable predictors are used)
- **Feature engineering** tailored to operational data:
  - Cyclic encodings for time variables (hour/day/month seasonality)
  - Smoothed historical-rate features (e.g., route/carrier “reliability” style signals) to stabilize sparse categories
- Comparative modeling of GBT vs. baselines, emphasizing robustness under skewed outcomes
- Evaluation beyond “one number,” including interpretability-oriented breakdowns and practical “risk” framing
- A **scaling analysis** demonstrating runtime behavior under:
  - **Scale-up:** increasing data fraction (10% → 25% → 50% → 100%)
  - **Scale-out:** increasing cluster capacity (worker nodes)

## 1.5 Report Structure

1. **Data Overview:** Source, timeline, and volume characteristics  
2. **Data Cleanup** Making the actual data file from various sources
3. **Exploratory Data Analysis (EDA):** Seasonality and operational patterns 
4. **ETL Pipeline:** Cleaning, filtering, and leakage prevention strategy  
5. **Feature Engineering:** Cyclic time representations and smoothed historical signals  
6. **Machine Learning Architecture:** GBT vs. baseline models in Spark MLlib  
7. **Evaluation:** Metrics, ablation comparisons, and (optional) clustering insights  
8. **Scaling Analysis:** Scale-up vs. scale-out performance and trade-offs  
9. **Conclusion & Future Work:** Summary, limitations, and extensions



## Dataset Cleanup

This demonstrates how the original 24 data files are combined into the 4 train and test sets provided.


# Notebook Preparation

In [137]:
import pandas as pd
import numpy as np
import time

import warnings
warnings.filterwarnings('ignore')

# Obtaining our Data

## About the Data

Our data comes from a variety of sources, all aimed at creating a full view of airport delay through various study metrics involving the airport, aircraft, airline, passengers, and weather.

Our primary dataset is the Bureau of Transportation Statistics' Montly On-Time Report, which for the year of 2019 comprises several million rows of data on every flight flown domestically for the entire year. We use and combine these monthly statistics with a variety of other data sets to gain further insights.

We use 5 informational datasets from the Bureau of Transportation Statistics:
* T3_AIR_CARRIER_SUMMARY_AIRPORT_ACTIVITY.csv
* B43_AIRCRAFT_INVENTORY.csv
* AIRPORT_COORDINATES.csv
* CARRIER_DECODE.csv
* P10_EMPLOYEES.csv


2 informational datasets from the National Centers for Environmental Information
* Airport_Weather.csv
* Airport_list.csv

The data sets can be refined at download, so I chose features that I needed when acquiring the data.

Our base data of on-time reporting is feature rich. We have detailed information for EVERY flight taken, including the date, the carrier, the tail number, the origin airport, the destination airport, the time the flight left, the reason for delay if delayed, the length of the flight, and the distance it traveled on the flight. We are interested in the delay and will clean for both general delay and specific delay.

In [138]:
# Load a month of data so we can see what kind of information we're working with
df = pd.read_csv('/content/ONTIME_REPORTING_01.csv')
df.shape

(583985, 33)

In [139]:
df.head()

,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,OP_UNIQUE_CARRIER,TAIL_NUM,OP_CARRIER_FL_NUM,ORIGIN_AIRPORT_ID,ORIGIN,ORIGIN_CITY_NAME,DEST_AIRPORT_ID,...,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,DISTANCE,DISTANCE_GROUP,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,Unnamed: 32
0,1,6,7,9E,N8694A,3280,10397,ATL,"Atlanta, GA",11150,...,47.0,37.0,83.0,1,NaN,NaN,NaN,NaN,NaN,NaN
1,1,7,1,9E,N8970D,3280,10397,ATL,"Atlanta, GA",11150,...,47.0,32.0,83.0,1,NaN,NaN,NaN,NaN,NaN,NaN
2,1,8,2,9E,N820AY,3280,10397,ATL,"Atlanta, GA",11150,...,47.0,39.0,83.0,1,NaN,NaN,NaN,NaN,NaN,NaN
3,1,9,3,9E,N840AY,3280,10397,ATL,"Atlanta, GA",11150,...,47.0,37.0,83.0,1,NaN,NaN,NaN,NaN,NaN,NaN
4,1,10,4,9E,N8969A,3280,10397,ATL,"Atlanta, GA",11150,...,47.0,41.0,83.0,1,NaN,NaN,NaN,NaN,NaN,NaN


In [140]:
# Check memory usage of this file
df.memory_usage().sum()
# These files are LARGE, and this is only one month.
# Part of our cleaning process will be to store our data in a more memory efficient manner.

np.int64(154172172)

In [141]:
# What types of data do we have? We can get a feel of what we may be able to reduce to a smaller integer
df.dtypes

,0
MONTH,int64
DAY_OF_MONTH,int64
DAY_OF_WEEK,int64
OP_UNIQUE_CARRIER,object
TAIL_NUM,object
OP_CARRIER_FL_NUM,int64
ORIGIN_AIRPORT_ID,int64
ORIGIN,object
ORIGIN_CITY_NAME,object
DEST_AIRPORT_ID,int64


In [142]:
# Descriptive stats for our data
df.describe()

,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,OP_CARRIER_FL_NUM,ORIGIN_AIRPORT_ID,DEST_AIRPORT_ID,CRS_DEP_TIME,DEP_TIME,DEP_DELAY_NEW,DEP_DEL15,...,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,DISTANCE,DISTANCE_GROUP,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,Unnamed: 32
count,583985.0,583985.000000,583985.000000,583985.000000,583985.000000,583985.000000,583985.000000,567633.000000,567630.000000,567630.000000,...,583851.000000,565963.000000,583985.000000,583985.000000,105222.000000,105222.000000,105222.000000,105222.000000,105222.000000,0.0
mean,1.0,15.960088,3.835626,2537.869334,12659.701982,12659.470015,1326.266642,1331.957814,13.258226,0.174281,...,143.557401,138.610077,797.742767,3.664861,20.106080,4.428551,18.508392,0.079508,24.950590,NaN
std,0.0,8.987942,1.921899,1821.736145,1519.405493,1519.336466,484.081000,495.404020,47.507390,0.379351,...,73.216303,73.078565,589.999261,2.323890,65.762202,38.535323,41.726392,3.203342,50.851973,NaN
min,1.0,1.000000,1.000000,1.000000,10135.000000,10135.000000,1.000000,1.000000,0.000000,0.000000,...,20.000000,16.000000,31.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN
25%,1.0,8.000000,2.000000,979.000000,11292.000000,11292.000000,917.000000,921.000000,0.000000,0.000000,...,90.000000,85.000000,363.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN
50%,1.0,16.000000,4.000000,2114.000000,12889.000000,12889.000000,1320.000000,1328.000000,0.000000,0.000000,...,125.000000,121.000000,640.000000,3.000000,0.000000,0.000000,5.000000,0.000000,0.000000,NaN
75%,1.0,24.000000,5.000000,3902.000000,13931.000000,13931.000000,1730.000000,1738.000000,5.000000,0.000000,...,175.000000,170.000000,1037.000000,5.000000,16.000000,0.000000,21.000000,0.000000,29.000000,NaN
max,1.0,31.000000,7.000000,7439.000000,16218.000000,16218.000000,2359.000000,2400.000000,1651.000000,1.000000,...,703.000000,737.000000,4983.000000,11.000000,1638.000000,1416.000000,1447.000000,816.000000,1514.000000,NaN


In [143]:
# Check our missing data
df.isna().sum()

,0
MONTH,0
DAY_OF_MONTH,0
DAY_OF_WEEK,0
OP_UNIQUE_CARRIER,0
TAIL_NUM,2543
OP_CARRIER_FL_NUM,0
ORIGIN_AIRPORT_ID,0
ORIGIN,0
ORIGIN_CITY_NAME,0
DEST_AIRPORT_ID,0


# Scrubbing/Cleaning our Data

## Loading Data for Merging

T3_AIR_CARRIER_SUMMARY_AIRPORT_ACTIVITY provides information on how many departures were performed (REV_ACRFT_DEP_PERF_510) and how many passengers were enplaned (REV_PAX_ENP_110) by CARRIER and AIRPORT. We'll use this data to provide metrics for the "busy-ness" of an airport and airline.

In [144]:
# Information on passenger activity for airports and airlines
passengers = pd.read_csv('/content/T3_AIR_CARRIER_SUMMARY_AIRPORT_ACTIVITY_2019.csv')
passengers

,OP_UNIQUE_CARRIER,CARRIER_NAME,ORIGIN_AIRPORT_ID,SERVICE_CLASS,REV_ACRFT_DEP_PERF_510,REV_PAX_ENP_110
0,04Q,Tradewind Aviation,15024,K,10.0,39.0
1,04Q,Tradewind Aviation,14843,K,677.0,3649.0
2,04Q,Tradewind Aviation,10257,V,4.0,6.0
3,04Q,Tradewind Aviation,15323,V,1.0,3.0
4,04Q,Tradewind Aviation,10158,V,1.0,2.0
...,...,...,...,...,...,...
27247,ZW,Air Wisconsin Airlines Corp,11637,K,122.0,4535.0
27248,ZW,Air Wisconsin Airlines Corp,11721,K,143.0,5800.0
27249,ZW,Air Wisconsin Airlines Corp,10469,K,248.0,8901.0
27250,ZW,Air Wisconsin Airlines Corp,12884,K,187.0,7923.0


B43_AIRCRAFT_INVENTORY provides information about specific tail numbers. We want to know the age of an aircraft, and how many passengers it seats.

In [145]:
# Manufacture year and passenger capacity for aircraft by unique aircraft tail number
aircraft = pd.read_csv("/content/B43_AIRCRAFT_INVENTORY.csv",encoding='latin1')
aircraft.drop_duplicates(subset='TAIL_NUM', inplace=True)
aircraft

,MANUFACTURE_YEAR,TAIL_NUM,NUMBER_OF_SEATS
0,1944,N54514,0.0
1,1945,N1651M,0.0
2,1953,N100CE,0.0
3,1953,N141FL,0.0
4,1953,N151FL,0.0
...,...,...,...
7378,2019,N14011,337.0
7379,2019,N16008,337.0
7380,2019,N16009,337.0
7381,2019,N2250U,276.0


AIRPORT_COORDINATES simply provides specific latitide/longitude for airports. We'll use this as location information.

In [146]:
# coordinates of airports
coords = pd.read_csv('/content/AIRPORT_COORDINATES.csv')
coords.drop_duplicates(subset='ORIGIN_AIRPORT_ID', inplace=True)
coords

,ORIGIN_AIRPORT_ID,DISPLAY_AIRPORT_NAME,LATITUDE,LONGITUDE
0,10001,Afognak Lake Airport,58.109444,-152.906667
1,10003,Bear Creek Mining Strip,65.548056,-161.071667
2,10004,Lik Mining Camp,68.083333,-163.166667
3,10005,Little Squaw Airport,67.570000,-148.183889
4,10006,Kizhuyak Bay,57.745278,-152.882778
...,...,...,...,...
18128,16908,Deer Park Airport,47.966944,-117.428611
18129,16909,South Texas International at Edinburg,26.441667,-98.122222
18130,16910,Louisa County Freeman Field,38.009722,-77.970000
18131,16911,Caldwell Industrial,43.641944,-116.635833


CARRIER_DECODE is to get a lookup table for airline codes to match into the main On-Time Reports.

In [147]:
# proper names of carriers for better EDA usage
names = pd.read_csv("/content/CARRIER_DECODE.csv")
names.drop_duplicates(inplace=True)
names.drop_duplicates(subset=['OP_UNIQUE_CARRIER'], inplace=True)
names

,AIRLINE_ID,OP_UNIQUE_CARRIER,CARRIER_NAME
0,21754,2PQ,21 Air LLC
3,20342,Q5,40-Mile Air
4,20342,WRB,40-Mile Air
6,19627,CIQ,A/S Conair
7,19072,AAE,AAA Airlines
...,...,...,...
2702,20379,ZKQ,Zantop International
2706,19771,ZAQ,Zas Airline Of Egypt
2707,21118,37,Zeal 320
2708,22069,ZG,ZIPAIR Tokyo Inc.


P10_EMPLOYEES is so we can determine how many employees a carrier has for Passenger Handling (flight attendants) as well as Ground Service, so that we can determine the employees per passenger.

In [148]:
# Employee statistics for carriers
employees = pd.read_csv('/content/P10_EMPLOYEES.csv')
employees = employees[['OP_UNIQUE_CARRIER', 'PASS_GEN_SVC_ADMIN', 'PASSENGER_HANDLING']]
employees = employees.groupby('OP_UNIQUE_CARRIER').sum().reset_index()
employees

,OP_UNIQUE_CARRIER,PASS_GEN_SVC_ADMIN,PASSENGER_HANDLING
0,0WQ,19,0
1,1BQ,41,0
2,2HQ,24,0
3,3EQ,32,0
4,5V,0,0
5,5X,0,0
6,5Y,273,0
7,8C,37,0
8,9E,1361,0
9,9S,3,0


### Cleaning Weather Data

Weather Data was acquired on a daily basis for each airport used in the data set. We will ultimately use snowfall, ground snow, precipitation, wind speed, and temperature as features in our data set.

In [149]:
# Weather report for top 90% of airport cities, in 2019
weather_report = pd.read_csv('/content/airport_weather_2019.csv')
weather_report

,STATION,NAME,DATE,AWND,PGTM,PRCP,SNOW,SNWD,TAVG,TMAX,...,WT08,WT09,WESD,WT10,PSUN,TSUN,SN32,SX32,TOBS,WT11
0,USW00013874,ATLANTA HARTSFIELD JACKSON INTERNATIONAL AIRPO...,1/1/2019,4.70,NaN,0.14,0.0,0.0,64.0,66.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,USW00013874,ATLANTA HARTSFIELD JACKSON INTERNATIONAL AIRPO...,1/2/2019,4.92,NaN,0.57,0.0,0.0,56.0,59.0,...,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,USW00013874,ATLANTA HARTSFIELD JACKSON INTERNATIONAL AIRPO...,1/3/2019,5.37,NaN,0.15,0.0,0.0,52.0,55.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,USW00013874,ATLANTA HARTSFIELD JACKSON INTERNATIONAL AIRPO...,1/4/2019,12.08,NaN,1.44,0.0,0.0,56.0,66.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,USW00013874,ATLANTA HARTSFIELD JACKSON INTERNATIONAL AIRPO...,1/5/2019,13.42,NaN,0.00,0.0,0.0,49.0,59.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38670,USW00093805,"TALLAHASSEE REGIONAL AIRPORT, FL US",2019-12-27,6.04,NaN,0.00,NaN,NaN,68.0,80.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
38671,USW00093805,"TALLAHASSEE REGIONAL AIRPORT, FL US",2019-12-28,5.37,NaN,0.06,NaN,NaN,69.0,74.0,...,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
38672,USW00093805,"TALLAHASSEE REGIONAL AIRPORT, FL US",2019-12-29,7.61,NaN,0.10,NaN,NaN,70.0,74.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
38673,USW00093805,"TALLAHASSEE REGIONAL AIRPORT, FL US",2019-12-30,5.82,NaN,0.02,NaN,NaN,68.0,72.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


We also have a set of the airports and their city names to match up with the weather dataset, so that we can then match this up to our On-Time Dataset

In [150]:
# Our list of cities and airports including the airport display name so that we can connect with our main df
cities = pd.read_csv('/content/airports_list.csv')
cities

,ORIGIN_AIRPORT_ID,DISPLAY_AIRPORT_NAME,ORIGIN_CITY_NAME,NAME
0,12992,Adams Field,"Little Rock, AR","NORTH LITTLE ROCK AIRPORT, AR US"
1,10257,Albany International,"Albany, NY","ALBANY INTERNATIONAL AIRPORT, NY US"
2,10140,Albuquerque International Sunport,"Albuquerque, NM","ALBUQUERQUE INTERNATIONAL AIRPORT, NM US"
3,10299,Anchorage International,"Anchorage, AK","ANCHORAGE TED STEVENS INTERNATIONAL AIRPORT, A..."
4,10397,Atlanta Municipal,"Atlanta, GA",ATLANTA HARTSFIELD JACKSON INTERNATIONAL AIRPO...
...,...,...,...,...
92,15370,Tulsa International,"Tulsa, OK","OKLAHOMA CITY WILL ROGERS WORLD AIRPORT, OK US"
93,12264,Washington Dulles International,"Washington, DC","WASHINGTON DULLES INTERNATIONAL AIRPORT, VA US"
94,13851,Will Rogers World,"Oklahoma City, OK","OKLAHOMA CITY WILL ROGERS WORLD AIRPORT, OK US"
95,12191,William P Hobby,"Houston, TX","HOUSTON WILLIAM P HOBBY AIRPORT, TX US"


In [151]:
# Connect our weather report with the city names
weather_merge = pd.merge(cities, weather_report, how='left', on='NAME')
weather_merge

,ORIGIN_AIRPORT_ID,DISPLAY_AIRPORT_NAME,ORIGIN_CITY_NAME,NAME,STATION,DATE,AWND,PGTM,PRCP,SNOW,...,WT08,WT09,WESD,WT10,PSUN,TSUN,SN32,SX32,TOBS,WT11
0,12992,Adams Field,"Little Rock, AR","NORTH LITTLE ROCK AIRPORT, AR US",USW00003952,2019-01-01,4.70,NaN,0.00,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,12992,Adams Field,"Little Rock, AR","NORTH LITTLE ROCK AIRPORT, AR US",USW00003952,2019-01-02,2.01,NaN,0.39,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,12992,Adams Field,"Little Rock, AR","NORTH LITTLE ROCK AIRPORT, AR US",USW00003952,2019-01-03,6.26,NaN,0.44,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,12992,Adams Field,"Little Rock, AR","NORTH LITTLE ROCK AIRPORT, AR US",USW00003952,2019-01-04,2.01,NaN,0.13,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,12992,Adams Field,"Little Rock, AR","NORTH LITTLE ROCK AIRPORT, AR US",USW00003952,2019-01-05,1.79,NaN,0.00,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35020,10713,Boise Air Terminal,"Boise, ID","BOISE AIR TERMINAL, ID US",USW00024131,2019-12-27,5.82,NaN,0.00,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
35021,10713,Boise Air Terminal,"Boise, ID","BOISE AIR TERMINAL, ID US",USW00024131,2019-12-28,2.24,NaN,0.00,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
35022,10713,Boise Air Terminal,"Boise, ID","BOISE AIR TERMINAL, ID US",USW00024131,2019-12-29,6.26,NaN,0.04,0.1,...,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
35023,10713,Boise Air Terminal,"Boise, ID","BOISE AIR TERMINAL, ID US",USW00024131,2019-12-30,2.46,NaN,0.00,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [152]:
# Get just the important metrics from the weather report (date, precipitation, snow, temp, wind)
weather = weather_merge[['DATE', 'PRCP', 'SNOW', 'SNWD', 'TMAX', 'AWND', 'ORIGIN_AIRPORT_ID']]

In [153]:
# Drop any rows where no weather was recorded
weather.drop(weather.loc[weather['ORIGIN_AIRPORT_ID'].isna()].index, axis=0, inplace=True)

In [154]:
# Look for null values in temperature
weather.loc[weather['TMAX'].isna()]

,DATE,PRCP,SNOW,SNWD,TMAX,AWND,ORIGIN_AIRPORT_ID
4786,2/11/2019,0.22,NaN,0.0,NaN,9.62,11298
19976,2019-10-06,0.00,NaN,0.0,NaN,12.30,15919
24068,12/23/2019,0.00,NaN,NaN,NaN,1.57,11066
24807,NaN,NaN,NaN,NaN,NaN,NaN,14843
30085,6/19/2019,NaN,NaN,0.0,NaN,7.61,14635
31953,8/1/2019,0.66,0.0,0.0,NaN,7.61,15304


In [155]:
# Impute mean in nan rows for temp and wind
weather['TMAX'].fillna(round(weather.groupby('ORIGIN_AIRPORT_ID')['TMAX'].transform('mean'), 1), inplace=True)
weather['AWND'].fillna(round(weather.groupby('ORIGIN_AIRPORT_ID')['AWND'].transform('mean'), 1), inplace=True)
weather.fillna(0, inplace=True)

In [156]:
# Check no NaN remain
weather.isna().sum()

,0
DATE,0
PRCP,0
SNOW,0
SNWD,0
TMAX,0
AWND,0
ORIGIN_AIRPORT_ID,0


In [157]:
# Cast data types to datetime so we can get the month and day of month to match up with main df
weather['DATE'] = pd.to_datetime(weather['DATE'], format='mixed')
weather['MONTH'] = pd.DatetimeIndex(weather['DATE']).month
weather['DAY_OF_MONTH'] = pd.DatetimeIndex(weather['DATE']).day
weather

,DATE,PRCP,SNOW,SNWD,TMAX,AWND,ORIGIN_AIRPORT_ID,MONTH,DAY_OF_MONTH
0,2019-01-01,0.00,0.0,0.0,45.0,4.70,12992,1,1
1,2019-01-02,0.39,0.0,0.0,39.0,2.01,12992,1,2
2,2019-01-03,0.44,0.0,0.0,41.0,6.26,12992,1,3
3,2019-01-04,0.13,0.0,0.0,47.0,2.01,12992,1,4
4,2019-01-05,0.00,0.0,0.0,62.0,1.79,12992,1,5
...,...,...,...,...,...,...,...,...,...
35020,2019-12-27,0.00,0.0,0.0,35.0,5.82,10713,12,27
35021,2019-12-28,0.00,0.0,0.0,39.0,2.24,10713,12,28
35022,2019-12-29,0.04,0.1,0.0,32.0,6.26,10713,12,29
35023,2019-12-30,0.00,0.0,0.0,34.0,2.46,10713,12,30


# Cleaning Function

## Main On-Time Reporting Data (ONTIME_REPORTING_XX)

TARGET VARIABLES 

* DEP_DEL15: Kept as target for Classification.

* DEP_DELAY_NEW: Kept as target for Regression.

### PREDICTORS (RETAINED):

* MONTH, DAY_OF_MONTH, DAY_OF_WEEK: Crucial for seasonality.

* CRS_DEP_TIME: CRITICAL. This is the scheduled departure time. Do not drop this. convert this to "Hour of Day" to capture morning/evening rush patterns.

* CRS_ARR_TIME: Useful to calculate scheduled duration, though less critical than departure time.

* CRS_ELAPSED_TIME: Keep. This represents the scheduled duration. Long flights behave differently than short ones.

* DISTANCE / DISTANCE_GROUP: Keep. Short vs. long-haul is a major factor.

* OP_UNIQUE_CARRIER / TAIL_NUM / ORIGIN_AIRPORT_ID / DEST_AIRPORT_ID: we need these to join other tables. For the final model, will likely replace these with "Target Encodings" or "One-Hot Encodings".

### LEAKAGE (Must Drop):

* DEP_TIME: This is the actual departure time. If we know this, we already know the delay.

* ARR_TIME: Actual arrival time. Leakage.

* ACTUAL_ELAPSED_TIME: Actual duration. Leakage.

* ARR_DELAY_NEW: If we know the arrival delay, we know the outcome.

* CANCELLED / CANCELLATION_CODE: If a flight is cancelled, it technically isn't "delayed" in the same way. Usually, these are filtered out or treated as a separate class.

* CARRIER_DELAY, WEATHER_DELAY, NAS_DELAY, SECURITY_DELAY, LATE_AIRCRAFT_DELAY: These explain why a delay happened after the fact. Cannot know these in advance.

### Feature Engineering:
1. Aircraft Inventory (B43_AIRCRAFT_INVENTORY)

MANUFACTURE_YEAR: Used to create a new feature called PLANE_AGE (Current Year - Manufacture Year). Older planes might have more mechanical delays. Once we create PLANE_AGE, we can drop MANUFACTURE_YEAR.

NUMBER_OF_SEATS: Keep. Larger planes (wide-body) might have different turnaround times than small regional jets.

2. Weather (airport_weather_xxxx)
Weather is often the biggest predictor of delay.

Keep All: PRCP (Precipitation), SNOW, SNWD (Snow Depth), TMAX (Temp), AWND (Wind).

Strategy: We must join this on DATE and ORIGIN_AIRPORT_ID.

3. Coordinates (AIRPORT_COORDINATES)
Keep: LATITUDE, LONGITUDE.

Usage: These are useful for clustering (K-Means) to group airports by region (e.g., "Northeast Corridor" vs "West Coast") or to calculate distances if they were missing.

### Summary Plan

Join everything into one master dataframe using the IDs (TAIL_NUM, AIRPORT_ID, DATE).

Create Features:

PLANE_AGE

DEP_HOUR (extracted from CRS_DEP_TIME)

IS_HOLIDAY (extracted from DATE)

Drop Leakage: DEP_TIME, ARR_TIME, ACTUAL_ELAPSED, Delay Reasons.

Drop Raw IDs (Last Step): Once we have encoded Carrier and Airport (using Target Encoding), then we can drop the raw OP_UNIQUE_CARRIER and ORIGIN_AIRPORT_ID columns before feeding data into the Machine Learning model.

In [173]:
def month_cleanup(monthly_data, aircraft, coords, names, weather, passengers, employees):

    '''Function which performs features engineering, data merges and cleanup using one month of On-Time data
    from Bureau of Transportation Services
    '''

    start = time.time()

    # ---------------------------------------------------------
    # 1. SANITIZATION (CRITICAL FOR DATA INTEGRITY)
    # ---------------------------------------------------------
    print("Sanitizing IDs, Dates, and Columns...")

    # FIX: Force to numeric first. Bad values become NaN.
    monthly_data['ORIGIN_AIRPORT_ID'] = pd.to_numeric(monthly_data['ORIGIN_AIRPORT_ID'], errors='coerce')
    monthly_data['MONTH'] = pd.to_numeric(monthly_data['MONTH'], errors='coerce')
    monthly_data['DAY_OF_MONTH'] = pd.to_numeric(monthly_data['DAY_OF_MONTH'], errors='coerce')

    # CRITICAL FIX: Drop rows where ID, Month, or Day became NaN *BEFORE* casting to int
    monthly_data.dropna(subset=['ORIGIN_AIRPORT_ID', 'MONTH', 'DAY_OF_MONTH'], inplace=True)

    # Now it is safe to cast to integers
    monthly_data['ORIGIN_AIRPORT_ID'] = monthly_data['ORIGIN_AIRPORT_ID'].astype('int64')
    monthly_data['MONTH'] = monthly_data['MONTH'].astype('int64')
    monthly_data['DAY_OF_MONTH'] = monthly_data['DAY_OF_MONTH'].astype('int64')

    # Force Time columns to numeric
    monthly_data['DEP_TIME'] = pd.to_numeric(monthly_data['DEP_TIME'], errors='coerce')
    monthly_data['CRS_DEP_TIME'] = pd.to_numeric(monthly_data['CRS_DEP_TIME'], errors='coerce')

    # ---------------------------------------------------------
    # 2. CLEANING
    # ---------------------------------------------------------
    print("Dropping NaNs from Dep Time, Tail Num. Dropping Cancellations.")
    monthly_data.dropna(subset=['DEP_TIME', 'TAIL_NUM'], inplace=True)
    monthly_data.drop(monthly_data.loc[monthly_data['CANCELLED']==1].index, axis=0, inplace=True)

    # ---------------------------------------------------------
    # 3. FEATURE ENGINEERING
    # ---------------------------------------------------------

    # --- SEGMENT NUMBER ---
    print("Adding Flight Number Sequence - SEGMENT_NUMBER...")
    monthly_data["SEGMENT_NUMBER"] = monthly_data.groupby(["TAIL_NUM", 'DAY_OF_MONTH'])["CRS_DEP_TIME"].rank("dense", ascending=True)
    monthly_data["SEGMENT_NUMBER"] = monthly_data["SEGMENT_NUMBER"].fillna(0).astype(int)

    # --- CONCURRENT FLIGHTS ---
    print("Adding Concurrent Flights - CONCURRENT_FLIGHTS")
    monthly_data['CONCURRENT_FLIGHTS'] = monthly_data.groupby(['ORIGIN_AIRPORT_ID','DAY_OF_MONTH', 'DEP_TIME_BLK'])['OP_UNIQUE_CARRIER'].transform("count")

    # --- DEP_HOUR ---
    print("Adding Scheduled Departure Hour - DEP_HOUR")
    monthly_data['DEP_HOUR'] = monthly_data['CRS_DEP_TIME'].fillna(0).astype(int).astype(str).str.zfill(4).str[:2].astype('int8')

    # ---------------------------------------------------------
    # 4. MERGING & IMPUTATION
    # ---------------------------------------------------------

    # --- SEATS ---
    print("Applying seat counts to flights - NUMBER_OF_SEATS")
    monthly_data = pd.merge(monthly_data, aircraft, how="left", on='TAIL_NUM')

    monthly_data['NUMBER_OF_SEATS'] = monthly_data['NUMBER_OF_SEATS'].fillna(monthly_data.groupby('OP_UNIQUE_CARRIER')['NUMBER_OF_SEATS'].transform('mean'))
    monthly_data['NUMBER_OF_SEATS'] = monthly_data['NUMBER_OF_SEATS'].fillna(monthly_data['NUMBER_OF_SEATS'].mean())
    monthly_data['NUMBER_OF_SEATS'] = pd.to_numeric(monthly_data['NUMBER_OF_SEATS'], errors='coerce').fillna(0).astype('int16')

    # --- CARRIER NAMES ---
    print("Applying Carrier Names - CARRIER_NAME")
    monthly_data = pd.merge(monthly_data, names, how='left', on=['OP_UNIQUE_CARRIER'])

    # --- FLIGHT STATS ---
    print("Adding flight statistics for carrier and airport...")
    monthly_data['AIRPORT_FLIGHTS_MONTH'] = monthly_data.groupby(['ORIGIN_AIRPORT_ID'])['ORIGIN_CITY_NAME'].transform('count')
    monthly_data['AIRLINE_FLIGHTS_MONTH'] = monthly_data.groupby(['OP_UNIQUE_CARRIER'])['ORIGIN_CITY_NAME'].transform('count')
    monthly_data['AIRLINE_AIRPORT_FLIGHTS_MONTH'] = monthly_data.groupby(['OP_UNIQUE_CARRIER', 'ORIGIN_AIRPORT_ID'])['ORIGIN_CITY_NAME'].transform('count')

    # --- PASSENGER STATS ---
    print("Adding passenger statistics for carrier and airport...")
    monthly_airport_passengers = pd.DataFrame(passengers.groupby(['ORIGIN_AIRPORT_ID'])['REV_PAX_ENP_110'].sum())
    monthly_data = pd.merge(monthly_data, monthly_airport_passengers, how='left', on=['ORIGIN_AIRPORT_ID'])

    monthly_data['REV_PAX_ENP_110'] = monthly_data['REV_PAX_ENP_110'].fillna(0)
    monthly_data['AVG_MONTHLY_PASS_AIRPORT'] = (monthly_data['REV_PAX_ENP_110']/12).astype('int64')

    monthly_airline_passengers = pd.DataFrame(passengers.groupby(['OP_UNIQUE_CARRIER'])['REV_PAX_ENP_110'].sum())
    monthly_data = pd.merge(monthly_data, monthly_airline_passengers, how='left', on=['OP_UNIQUE_CARRIER'], suffixes=('_x', '_y'))

    monthly_data['REV_PAX_ENP_110_y'] = monthly_data['REV_PAX_ENP_110_y'].fillna(0)
    monthly_data['AVG_MONTHLY_PASS_AIRLINE'] = (monthly_data['REV_PAX_ENP_110_y']/12).astype('int64')

    # --- EMPLOYEE STATS ---
    print("Adding employee statistics for carrier...")
    monthly_data = pd.merge(monthly_data, employees, how='left', on=['OP_UNIQUE_CARRIER'])
    monthly_data['FLT_ATTENDANTS_PER_PASS'] = monthly_data['PASSENGER_HANDLING']/monthly_data['REV_PAX_ENP_110_y']
    monthly_data['GROUND_SERV_PER_PASS'] = monthly_data['PASS_GEN_SVC_ADMIN']/monthly_data['REV_PAX_ENP_110_y']

    # --- PLANE AGE ---
    print("Calculate Fleet Age - PLANE_AGE")
    monthly_data['MANUFACTURE_YEAR'] = monthly_data['MANUFACTURE_YEAR'].fillna(monthly_data.groupby('OP_UNIQUE_CARRIER')['MANUFACTURE_YEAR'].transform('mean'))
    monthly_data['MANUFACTURE_YEAR'] = monthly_data['MANUFACTURE_YEAR'].fillna(monthly_data['MANUFACTURE_YEAR'].mean())
    monthly_data['PLANE_AGE'] = 2019 - monthly_data['MANUFACTURE_YEAR']

    # --- COORDINATES ---
    print("Adding airport coordinates - LATITUDE, LONGITUDE, DEPARTING_AIRPORT")
    monthly_data = pd.merge(monthly_data, coords, how='left', on=['ORIGIN_AIRPORT_ID'])
    monthly_data['LATITUDE'] = round(monthly_data['LATITUDE'], 3)
    monthly_data['LONGITUDE'] = round(monthly_data['LONGITUDE'], 3)

    # --- PREVIOUS AIRPORT ---
    print("Adding airports - PREVIOUS_AIRPORT")
    monthly_data.sort_values(by='SEGMENT_NUMBER', inplace=True)

    segment_temp = monthly_data[['DAY_OF_MONTH', 'TAIL_NUM', 'DISPLAY_AIRPORT_NAME', 'SEGMENT_NUMBER']].copy()
    segment_temp.sort_values(by='SEGMENT_NUMBER', inplace=True)

    monthly_data = pd.merge_asof(
        monthly_data,
        segment_temp,
        on='SEGMENT_NUMBER',
        by=['DAY_OF_MONTH', 'TAIL_NUM'],
        allow_exact_matches=False
    )
    monthly_data['DISPLAY_AIRPORT_NAME_y'].fillna('NONE', inplace=True)
    monthly_data.rename(columns={"DISPLAY_AIRPORT_NAME_y": "PREVIOUS_AIRPORT", "DISPLAY_AIRPORT_NAME_x": "DEPARTING_AIRPORT"}, inplace=True)

    # ---------------------------------------------------------
    # 5. FINAL CLEANUP
    # ---------------------------------------------------------
    print("Dropping bottom 10% of airports")
    monthly_data.drop(monthly_data.loc[monthly_data['AIRPORT_FLIGHTS_MONTH'] < 1100].index, axis=0, inplace=True)

    # --- WEATHER ---
    print("Adding daily weather data...")
    monthly_data = pd.merge(monthly_data, weather, how='left', on=['ORIGIN_AIRPORT_ID', 'MONTH', 'DAY_OF_MONTH'])

    # --- FINAL COLUMN DROP ---
    print("Clean up unneeded columns")
    monthly_data.drop(columns = [
                   'CANCELLED', 'CANCELLATION_CODE',
                   'CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY', 'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY',
                  'Unnamed: 32',  'ARR_TIME_BLK', 'DEST_CITY_NAME',  'OP_CARRIER_FL_NUM',  'OP_UNIQUE_CARRIER',
                       'AIRLINE_ID',
                     'ORIGIN_CITY_NAME',  'PASSENGER_HANDLING', 'REV_PAX_ENP_110_x', 'REV_PAX_ENP_110_y',
                                 'PASS_GEN_SVC_ADMIN'
                                 ],
                    axis=1, inplace=True)

    # --- TYPE CASTING ---
    print("Cleaning up data types")
    monthly_data['MONTH'] = monthly_data['MONTH'].astype('object')
    monthly_data['DAY_OF_WEEK'] = monthly_data['DAY_OF_WEEK'].astype('object')

    monthly_data['DEP_DEL15'] = pd.to_numeric(monthly_data['DEP_DEL15'], errors='coerce').fillna(0).astype('int8')
    monthly_data['DISTANCE_GROUP'] = pd.to_numeric(monthly_data['DISTANCE_GROUP'], errors='coerce').fillna(0).astype('int8')

    monthly_data['AIRPORT_FLIGHTS_MONTH'] = monthly_data['AIRPORT_FLIGHTS_MONTH'].astype('int64')
    monthly_data['AIRLINE_FLIGHTS_MONTH'] = monthly_data['AIRLINE_FLIGHTS_MONTH'].astype('int64')
    monthly_data['AIRLINE_AIRPORT_FLIGHTS_MONTH'] = monthly_data['AIRLINE_AIRPORT_FLIGHTS_MONTH'].astype('int64')
    monthly_data['PLANE_AGE'] = pd.to_numeric(monthly_data['PLANE_AGE'], errors='coerce').fillna(0).astype('int32')

    monthly_data.reset_index(inplace=True, drop=True)

    print(f'Elapsed Time: {time.time() - start}')
    print("FINISHED")

    return monthly_data

# Process Train/Test Sets

This function processes our data set and combines it into one large master set.

In [174]:
import pandas as pd
import gc  # Garbage Collector to free up RAM

# Initialize list to hold processed frames
frames = []

# Loop through months 1 to 12
for i in range(1, 13):
    month_str = str(i).zfill(2) # Creates "01", "02", ... "12"

    #  UPDATE THIS PATH to match your Google Colab location
    # If files are in the main folder, remove '/content/'
    file_path = f'/content/ONTIME_REPORTING_{month_str}.csv'

    print(f"Processing {file_path}...")

    try:
        # Load raw file
        # FIX ADDED: on_bad_lines='skip' ignores corrupted rows
        # FIX ADDED: low_memory=False helps processing large mixed-type files
        df = pd.read_csv(file_path, on_bad_lines='skip', low_memory=False)

        # Run cleanup function
        df_clean = month_cleanup(df, aircraft, coords, names, weather, passengers, employees)

        # Append to list
        frames.append(df_clean)

        # Clean up memory immediately to prevent Colab crash
        del df
        del df_clean
        gc.collect()

    except FileNotFoundError:
        print(f" Error: Could not find {file_path}. Please check your file path.")

# Combine all months
if frames:
    print("Concatenating all months...")
    all_data = pd.concat(frames, ignore_index=True)

    # Clean up list memory
    del frames
    gc.collect()

    # Save files
    print("Saving to pickle and csv...")
    # Saving to current directory in Colab so you can see them easily
    all_data.to_pickle("train_val.pkl")
    all_data.to_csv('train_val.csv', index=False)
    print("Done! Files saved.")
else:
    print("No data was processed.")

Processing /content/ONTIME_REPORTING_01.csv...
Sanitizing IDs, Dates, and Columns...
Dropping NaNs from Dep Time, Tail Num. Dropping Cancellations.
Adding Flight Number Sequence - SEGMENT_NUMBER...
Adding Concurrent Flights - CONCURRENT_FLIGHTS
Adding Scheduled Departure Hour - DEP_HOUR
Applying seat counts to flights - NUMBER_OF_SEATS
Applying Carrier Names - CARRIER_NAME
Adding flight statistics for carrier and airport...
Adding passenger statistics for carrier and airport...
Adding employee statistics for carrier...
Calculate Fleet Age - PLANE_AGE
Adding airport coordinates - LATITUDE, LONGITUDE, DEPARTING_AIRPORT
Adding airports - PREVIOUS_AIRPORT
Dropping bottom 10% of airports
Adding daily weather data...
Clean up unneeded columns
Cleaning up data types
Elapsed Time: 3.477111339569092
FINISHED
Processing /content/ONTIME_REPORTING_02.csv...
Sanitizing IDs, Dates, and Columns...
Dropping NaNs from Dep Time, Tail Num. Dropping Cancellations.
Adding Flight Number Sequence - SEGMENT_N

In [175]:
from google.colab import drive
import shutil

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Copy files to your Drive (Change 'My Drive' to a specific folder if you want)
print("Copying to Google Drive...")
shutil.copy("train_val.pkl", "/content/drive/My Drive/train_val.pkl")
shutil.copy("train_val.csv", "/content/drive/My Drive/train_val.csv")
print("Saved to Google Drive!")

Mounted at /content/drive
Copying to Google Drive...
Saved to Google Drive!


In [187]:
import pandas as pd

# Load the pickle file (it's much faster than CSV)
df = pd.read_pickle("train_val.pkl")

# Check the shape (Rows, Columns)
print(f"Dataset Shape: {df.shape}")

# Look at the first 5 rows
print("\nFirst 5 Rows:")
display(df.head())

# Check for any remaining Null values
print("\nMissing Values:")
print(df.isna().sum())

# Check your Target Variable Balance (0 = On Time, 1 = Delayed)
print("\nTarget Class Balance (DEP_DEL15):")
print(df['DEP_DEL15'].value_counts(normalize=True))

Dataset Shape: (7097476, 35)

First 5 Rows:


,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,TAIL_NUM,DEST_AIRPORT_ID,CRS_DEP_TIME,DEP_DEL15,DEP_TIME_BLK,CRS_ARR_TIME,CRS_ELAPSED_TIME,...,DEPARTING_AIRPORT,LATITUDE,LONGITUDE,PREVIOUS_AIRPORT,DATE,PRCP,SNOW,SNWD,TMAX,AWND
0,1,30,3,N326NB,11433,605.0,0,0600-0659,744,99.0,...,Pittsburgh International,40.497,-80.236,NONE,2019-01-30,0.0,0.0,0.0,16.0,13.20
1,1,30,3,N776DE,11433,910.0,0,0900-0959,1647,277.0,...,Los Angeles International,33.942,-118.408,NONE,2019-01-30,0.0,0.0,0.0,64.0,4.70
2,1,30,3,N892AT,13487,815.0,0,0800-0859,1021,186.0,...,Douglas Municipal,35.219,-80.936,NONE,2019-01-30,0.0,0.0,0.0,47.0,7.38
3,1,30,3,N549US,10397,620.0,0,0600-0659,1353,273.0,...,Seattle International,47.447,-122.306,NONE,2019-01-30,0.0,0.0,0.0,51.0,2.46
4,1,30,3,N322DN,14869,600.0,0,0600-0659,851,111.0,...,Los Angeles International,33.942,-118.408,NONE,2019-01-30,0.0,0.0,0.0,64.0,4.70



Missing Values:
MONTH                                0
DAY_OF_MONTH                         0
DAY_OF_WEEK                          0
TAIL_NUM                             0
DEST_AIRPORT_ID                      0
CRS_DEP_TIME                         8
DEP_DEL15                            0
DEP_TIME_BLK                         5
CRS_ARR_TIME                         8
CRS_ELAPSED_TIME                    20
DISTANCE                            23
DISTANCE_GROUP                       0
SEGMENT_NUMBER                       0
CONCURRENT_FLIGHTS                   5
DEP_HOUR                             0
NUMBER_OF_SEATS                      0
CARRIER_NAME                         0
AIRPORT_FLIGHTS_MONTH                0
AIRLINE_FLIGHTS_MONTH                0
AIRLINE_AIRPORT_FLIGHTS_MONTH        0
AVG_MONTHLY_PASS_AIRPORT             0
AVG_MONTHLY_PASS_AIRLINE             0
FLT_ATTENDANTS_PER_PASS              0
GROUND_SERV_PER_PASS                 0
PLANE_AGE                            0
DEPARTIN

In [188]:
# 1. Inspect original counts
print("Original Class Counts:")
print(all_data['DEP_DEL15'].value_counts())

# 2. Filter data to keep ONLY valid 0 (On Time) and 1 (Delayed) classes
# This removes any rare garbage values (like 2, 99, etc.) that appear only once
all_data = all_data[all_data['DEP_DEL15'].isin([0, 1])]

# 3. Confirm cleaning
print("\nCleaned Class Counts:")
print(all_data['DEP_DEL15'].value_counts())

Original Class Counts:
DEP_DEL15
0    5743013
1    1354451
Name: count, dtype: int64

Cleaned Class Counts:
DEP_DEL15
0    5743013
1    1354451
Name: count, dtype: int64


## Data Split - Test and Validation

We need to split our data into a train and testing set, because we will be using target encoding on our data. It's important that our target encoding not utilize any information found in either our Validation or our Test sets of new, unseen data. In fact, we use the target encoding that is created in our Train set to populate our Val and Test set target encodings, ensuring that our Validation/Test sets contain no leakage.

In [189]:
from sklearn.model_selection import train_test_split

# Split into subsets with Stratification
# Stratify ensures the % of delays is the same in both Train and Test
train, test = train_test_split(all_data, test_size=.3, random_state=42, stratify=all_data['DEP_DEL15'])

# Verify the split
print(f"Train Shape: {train.shape}")
print(f"Test Shape: {test.shape}")

# Show the first few rows of the Train set
train.head()

Train Shape: (4968224, 35)
Test Shape: (2129240, 35)


,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,TAIL_NUM,DEST_AIRPORT_ID,CRS_DEP_TIME,DEP_DEL15,DEP_TIME_BLK,CRS_ARR_TIME,CRS_ELAPSED_TIME,...,DEPARTING_AIRPORT,LATITUDE,LONGITUDE,PREVIOUS_AIRPORT,DATE,PRCP,SNOW,SNWD,TMAX,AWND
5931780,11,21,4,N717SA,15304,1950.0,0,1900-1959,2205,135.0,...,Indianapolis Muni/Weir Cook,39.729,-86.282,William P Hobby,2019-11-21,0.08,0.0,0.0,52.0,13.65
6728269,12,6,5,N418SW,10397,1234.0,1,1200-1259,1523,109.0,...,Shreveport Regional,32.447,-93.826,Atlanta Municipal,NaT,NaN,NaN,NaN,NaN,NaN
5491946,11,12,2,N110SY,12266,715.0,0,0700-0759,830,75.0,...,Dallas Fort Worth Regional,32.894,-97.030,NONE,2019-11-12,0.00,0.0,0.0,39.0,11.41
1183530,3,8,5,N955UW,12953,1200.0,0,1200-1259,1321,81.0,...,Logan International,42.364,-71.006,Philadelphia International,2019-03-08,0.00,0.0,0.0,38.0,10.96
6457496,12,5,4,N345NB,11775,1950.0,0,1900-1959,2132,162.0,...,Atlanta Municipal,33.641,-84.427,Roanoke Regional/Woodrum Field,2019-12-05,0.00,0.0,0.0,65.0,5.82


In [190]:
# Our Validation set
test.head()

,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,TAIL_NUM,DEST_AIRPORT_ID,CRS_DEP_TIME,DEP_DEL15,DEP_TIME_BLK,CRS_ARR_TIME,CRS_ELAPSED_TIME,...,DEPARTING_AIRPORT,LATITUDE,LONGITUDE,PREVIOUS_AIRPORT,DATE,PRCP,SNOW,SNWD,TMAX,AWND
2036300,4,12,5,N422WN,13796,2130.0,0,2100-2159,2245,75.0,...,Los Angeles International,33.942,-118.408,McCarran International,2019-04-12,0.00,0.0,0.0,73.0,12.30
4983032,10,10,4,N335NB,12892,600.0,0,0600-0659,815,135.0,...,Portland International,45.589,-122.595,NONE,2019-10-10,0.00,0.0,0.0,62.0,10.29
5705515,11,1,5,N408AS,12892,1545.0,0,1500-1559,1830,165.0,...,Seattle International,47.447,-122.306,Anchorage International,2019-11-01,0.00,0.0,0.0,56.0,10.74
5291660,10,8,2,N131EV,14492,2030.0,1,2000-2059,2226,116.0,...,LaGuardia,40.779,-73.876,Kansas City International,2019-10-08,0.03,0.0,0.0,66.0,12.53
1764381,4,24,3,N250WN,11292,925.0,0,0900-0959,1245,140.0,...,Sacramento International,38.696,-121.590,Los Angeles International,2019-04-24,0.00,0.0,0.0,90.0,3.80


## Target Encoding

For our target encoding, we will group monthly delay statistics by the following categories:

- Carrier
- Airport (Use for both departing airport and arriving airport)
- Day of Week
- Departure Block

In [191]:
# 1. Calculate Global Mean (Baseline probability of delay)
global_mean = train['DEP_DEL15'].mean()
print(f"Global Delay Mean: {global_mean:.4f}")

def get_smoothed_stats(df, group_cols, target_col='DEP_DEL15', weight=10):
    '''
    Calculates smoothed target encoding.
    weight (m): The "smoothing factor". Higher = trust the global mean more for small samples.
    '''
    # Calculate count and mean
    stats = df.groupby(group_cols)[target_col].agg(['count', 'mean'])

    # Apply Smoothing Formula
    # (n * mean + m * global_mean) / (n + m)
    smoothed = (stats['count'] * stats['mean'] + weight * global_mean) / (stats['count'] + weight)

    return smoothed.reset_index(name='SMOOTHED_MEAN')

# --- 1. Carrier Lookup ---
carrier_historical = get_smoothed_stats(train, ['CARRIER_NAME', 'MONTH'])
carrier_historical.rename(columns={'SMOOTHED_MEAN':'CARRIER_HISTORICAL'}, inplace=True)

# --- 2. Airport Lookup ---
# Statistics for delays departing from an airport
airport_historical = get_smoothed_stats(train, ['DEPARTING_AIRPORT', 'MONTH'])
airport_historical.rename(columns={'SMOOTHED_MEAN':'DEP_AIRPORT_HIST'}, inplace=True)

# --- 3. Previous Airport Lookup (CRITICAL FIX) ---
# We use .copy() so we don't break the original airport_historical dataframe
prev_airport_historical = airport_historical.copy()
# We map the stats of 'DEPARTING_AIRPORT' to 'PREVIOUS_AIRPORT'
# (i.e. if the previous airport was JFK, how often are flights FROM JFK delayed?)
prev_airport_historical.rename(columns={
    'DEPARTING_AIRPORT': 'PREVIOUS_AIRPORT',
    'DEP_AIRPORT_HIST': 'PREV_AIRPORT_HIST'
}, inplace=True)

# --- 4. Day of Week Lookup ---
day_historical = get_smoothed_stats(train, ['DAY_OF_WEEK', 'MONTH'])
day_historical.rename(columns={'SMOOTHED_MEAN':'DAY_HISTORICAL'}, inplace=True)

# --- 5. Departure Block Lookup ---
dep_block_lookup = get_smoothed_stats(train, ['DEP_TIME_BLK', 'MONTH'])
dep_block_lookup.rename(columns={'SMOOTHED_MEAN':'DEP_BLOCK_HIST'}, inplace=True)

print("Lookup tables created with Smoothing!")

Global Delay Mean: 0.1908
Lookup tables created with Smoothing!


In [192]:
# -----------------------------------------------------------------------------
# 1. MERGE LOOKUP TABLES (TRAIN)
# -----------------------------------------------------------------------------
print("Merging lookup tables to TRAIN...")
train = pd.merge(train, carrier_historical, how='left', on=['CARRIER_NAME', 'MONTH'])
train = pd.merge(train, airport_historical, how='left', on=['DEPARTING_AIRPORT', 'MONTH'])
train = pd.merge(train, prev_airport_historical, how='left', on=['PREVIOUS_AIRPORT', 'MONTH'])
train = pd.merge(train, day_historical, how='left', on=['DAY_OF_WEEK', 'MONTH'])
train = pd.merge(train, dep_block_lookup, how='left', on=['DEP_TIME_BLK', 'MONTH'])

# -----------------------------------------------------------------------------
# 2. MERGE LOOKUP TABLES (TEST)
# -----------------------------------------------------------------------------
print("Merging lookup tables to TEST...")
test = pd.merge(test, carrier_historical, how='left', on=['CARRIER_NAME', 'MONTH'])
test = pd.merge(test, airport_historical, how='left', on=['DEPARTING_AIRPORT', 'MONTH'])
test = pd.merge(test, prev_airport_historical, how='left', on=['PREVIOUS_AIRPORT', 'MONTH'])
test = pd.merge(test, day_historical, how='left', on=['DAY_OF_WEEK', 'MONTH'])
test = pd.merge(test, dep_block_lookup, how='left', on=['DEP_TIME_BLK', 'MONTH'])

# -----------------------------------------------------------------------------
# 3. FILL MISSING VALUES (CRITICAL)
# -----------------------------------------------------------------------------
# We use the 'global_mean' (approx 0.1908) calculated in the previous block.
# This ensures that if a Carrier/Airport/Time was not seen in the training data,
# the model assumes the baseline probability of delay rather than crashing.

new_columns = [
    'CARRIER_HISTORICAL',
    'DEP_AIRPORT_HIST',
    'PREV_AIRPORT_HIST',
    'DAY_HISTORICAL',
    'DEP_BLOCK_HIST'
]

print(f"Filling NaNs with Global Mean: {global_mean:.4f}")

for col in new_columns:
    train[col] = train[col].fillna(global_mean)
    test[col] = test[col].fillna(global_mean)



Merging lookup tables to TRAIN...
Merging lookup tables to TEST...
Filling NaNs with Global Mean: 0.1908


In [194]:

# SAVE FILES TO GOOGLE DRIVE

drive_path = "/content/drive/My Drive/"

print(f"Saving files to {drive_path}...")

try:
    train.to_pickle(drive_path + "train.pkl")
    train.to_csv(drive_path + "train.csv", index=False)

    test.to_pickle(drive_path + "test.pkl")
    test.to_csv(drive_path + "test.csv", index=False)

    print("Success! Train and Test sets are saved to your Google Drive.")
except OSError:
    print("Error: Could not save to Drive. Make sure you accepted the authentication prompt.")

Saving files to /content/drive/My Drive/...
Success! Train and Test sets are saved to your Google Drive.
